# backward-on-scalar-loss composite — cx19: backward on scalar loss, then maybe fire the eval callback

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backward-on-scalar-loss`, `log-samples-eval-callback`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "backward-on-scalar-loss"
DD_ATOM_IDS = ["backward-on-scalar-loss", "log-samples-eval-callback"]
DD_SUBTOPICS = ["PyTorch: backward()", "Logging: log-samples eval callback"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A real VAE/GAN training step has two responsibilities per micro-batch:

1. **Backward pass on the scalar loss.** Per-sample losses must be **reduced** to a 0-dim scalar before `.backward()`. Calling `.backward()` on a vector raises `RuntimeError: grad can be implicitly created only for scalar outputs`. The canonical reduction is `.mean()` (or `.sum()` / explicit weights). Atom: `backward-on-scalar-loss`.
2. **Conditional eval/logging callback.** Every `K` steps, the trainer calls a callback that generates `N` samples (e.g. decoder draws from the prior) and writes them to a sink. Atom: `log-samples-eval-callback`.

**Why compose them.** The most common bug at this seam is calling `.backward()` on the *unreduced* per-sample loss because the eval callback was tacked on AFTER the loop body without a `.mean()`. The composition forces both steps to happen in the right order:

```python
per_sample_loss = recon_loss + kl_loss   # shape (B,)
loss = per_sample_loss.mean()            # backward-on-scalar-loss.
loss.backward()
if step % log_every == 0:                # log-samples-eval-callback.
    samples = sample_fn()
    sink.append(samples)
```

We test that `.backward()` is invoked on a scalar (autograd raises otherwise) and that the callback fires exactly on the documented schedule.

### Composite Exercise — backward on scalar loss, then maybe fire the eval callback

**Atoms exercised together**: `backward-on-scalar-loss`, `log-samples-eval-callback`

Implement `cx19_step_then_maybe_log(per_sample_loss, param, log_every, step, sample_fn, sink)`.

Inputs:
- `per_sample_loss` — a `t.Tensor` of shape `(B,)`, where each entry is the loss for one sample in the batch. Has `requires_grad=True` (descends from `param`).
- `param` — a leaf `t.Tensor` with `requires_grad=True`, which `per_sample_loss` was built from. The function MUST populate `param.grad`.
- `log_every` — int. Fire the callback iff `step % log_every == 0`.
- `step` — int, the current global step number.
- `sample_fn` — a callable `() -> Any`. Call it to obtain a sample batch.
- `sink` — a `list`. When the callback fires, append the sample-fn output AND the step number as a `(step, sample)` tuple.

Required behaviour, in this order:
1. Reduce `per_sample_loss` to a **scalar** via `.mean()` (atom: backward-on-scalar-loss).
2. Call `.backward()` on that scalar. After this, `param.grad` is populated.
3. If `step % log_every == 0`, call `sample_fn()` and append `(step, sample)` to `sink` (atom: log-samples-eval-callback).
4. Return the scalar loss value (a Python float).

The test confirms:
- `param.grad` is populated (NOT `None`) after the call.
- Callback fires exactly when `step % log_every == 0` — not on other steps.
- Sink entries are `(step, sample_fn_output)` tuples in step order.
- Calling `.backward()` on the unreduced vector would raise — your reduction is load-bearing.

In [ ]:
def cx19_step_then_maybe_log(per_sample_loss, param, log_every, step, sample_fn, sink):
    # Atom A (backward-on-scalar-loss): reduce per-sample loss to a 0-dim scalar.
    loss = per_sample_loss.mean()
    loss.backward()
    # Atom B (log-samples-eval-callback): fire every `log_every` steps.
    if step % log_every == 0:
        sample = sample_fn()
        sink.append((step, sample))
    return loss.item()


<details><summary>Show solution — cx19</summary>

```python
def cx19_step_then_maybe_log(per_sample_loss, param, log_every, step, sample_fn, sink):
    # Atom A (backward-on-scalar-loss): reduce per-sample loss to a 0-dim scalar.
    loss = per_sample_loss.mean()
    loss.backward()
    # Atom B (log-samples-eval-callback): fire every `log_every` steps.
    if step % log_every == 0:
        sample = sample_fn()
        sink.append((step, sample))
    return loss.item()
```

The classic bug here is calling `per_sample_loss.backward()` directly — autograd needs an explicit `gradient=` argument for non-scalar outputs, and the natural default is `.mean()`. The callback is a clean side-channel: it doesn't read `loss` itself, only the step counter — so the order doesn't matter for correctness, but doing backward FIRST matches how trainers are written (backward → optional callbacks → optimizer step).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["PyTorch: backward()", "Logging: log-samples eval callback"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()